# 📊 Instagram Follows — Analysis Notebook

Complete analysis of the CSV generated by `analyze_follows.py`.

**Goals:**
- Identify accounts to unfollow (inactive, ghosts, low engagement)
- Segment follows (personal / business / verified)
- Cluster by bio thematics
- Detect anomalies (suspicious bots, fake accounts)

**Requirements:** `pip install pandas matplotlib seaborn`

## 1. Load & preview

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from config import SCAN_CSV

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

df = pd.read_csv(SCAN_CSV, encoding='utf-8-sig')
print(f'📊 {len(df)} follows loaded')
print(f'📋 Columns : {list(df.columns)}')
df.head()

In [ ]:
print('=== STATUS DISTRIBUTION ===')
print(df['status'].value_counts())
print()
print('=== ACCOUNT TYPES ===')
print(f"Private (accessible)     : {df['is_private'].sum()}")
print(f"Verified                 : {df['is_verified'].sum()}")
print(f"Business                 : {df['is_business'].sum()}")
print(f"With highlights          : {df['has_highlights'].sum()}")
print(f"With public story        : {df['has_public_story'].sum()}")

## 2. Inactivity distribution

In [ ]:
df_ok = df[df['status'] == 'OK'].copy()
df_ok['days_since_last_post'] = pd.to_numeric(df_ok['days_since_last_post'], errors='coerce')
df_ok = df_ok.dropna(subset=['days_since_last_post'])

bins = [0, 7, 30, 90, 180, 365, 730, 99999]
labels = ['🔥 7d', '📱 30d', '📅 90d', '⚠️ 6 months', '🚨 1 year', '💀 2 years', '☠️ 2+ years']
df_ok['inactivity_bucket'] = pd.cut(df_ok['days_since_last_post'], bins=bins, labels=labels)

counts = df_ok['inactivity_bucket'].value_counts().reindex(labels)
print('=== INACTIVITY DISTRIBUTION ===')
for label, count in counts.items():
    pct = 100 * count / counts.sum() if counts.sum() > 0 else 0
    print(f'{label:15s} : {count:4d} accounts ({pct:.1f}%)')

fig, ax = plt.subplots()
counts.plot(kind='bar', ax=ax, color=['#2ecc71','#27ae60','#f39c12','#e67e22','#e74c3c','#c0392b','#7f0000'])
ax.set_title('Follows by inactivity', fontsize=14, fontweight='bold')
ax.set_xlabel('Last post ago')
ax.set_ylabel('Number of accounts')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 3. 🎯 Top unfollow candidates

Composite score: inactive + no story + no highlight + weak ER = dead account.

In [ ]:
def unfollow_score(row):
    """0 to 100 score based ONLY on the account's own activity."""
    if row['status'] != 'OK':
        return 0
    score = 0
    days = row['days_since_last_post']
    if pd.notna(days):
        if days > 730: score += 60
        elif days > 365: score += 45
        elif days > 180: score += 30
        elif days > 90: score += 15
    if not row['has_public_story']: score += 15
    if not row['has_highlights']: score += 15
    er = row.get('last_post_engagement_rate_pct')
    if pd.notna(er) and er < 0.1: score += 10
    return min(score, 100)

df['score_unfollow'] = df.apply(unfollow_score, axis=1)

print('=== TOP 30 UNFOLLOW CANDIDATES ===')
top_unfollow = df[df['status']=='OK'].sort_values('score_unfollow', ascending=False).head(30)
cols_show = ['username','full_name','days_since_last_post','followers','last_post_engagement_rate_pct','has_highlights','has_public_story','score_unfollow']
top_unfollow[cols_show]

In [ ]:
never = df[df['status'] == 'NEVER_POSTED']
print(f'💀 {len(never)} accounts have NEVER posted (obvious unfollow) :')
never[['username','full_name','followers']].head(30)

## 4. 🔍 Segmentation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

df_ok['is_verified'].value_counts().plot(kind='pie', ax=axes[0], autopct='%1.1f%%',
    labels=['Not verified','Verified'], colors=['#95a5a6','#3498db'])
axes[0].set_title('Verified')
axes[0].set_ylabel('')

df_ok['is_business'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
    labels=['Personal','Business'], colors=['#f39c12','#e74c3c'])
axes[1].set_title('Personal vs Business')
axes[1].set_ylabel('')

df_ok['is_private'].value_counts().plot(kind='pie', ax=axes[2], autopct='%1.1f%%',
    labels=['Public','Private'], colors=['#2ecc71','#9b59b6'])
axes[2].set_title('Public vs Private')
axes[2].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
biz = df[(df['is_business']==True) & (df['business_category'].notna()) & (df['business_category']!='')]
if len(biz) > 0:
    top_cat = biz['business_category'].value_counts().head(15)
    fig, ax = plt.subplots()
    top_cat.plot(kind='barh', ax=ax, color='#e74c3c')
    ax.set_title('Top 15 business categories in follows', fontweight='bold')
    ax.set_xlabel('Number of accounts')
    plt.tight_layout()
    plt.show()
else:
    print('No exploitable business category.')

## 5. 🏷️ Thematic clustering by bio

Feel free to enrich the `themes` dictionary below with keywords in your own language.

In [ ]:
themes = {
    'Sports ⛳':     ['golf','tennis','football','soccer','swing','birdie','fairway','gym','fitness','yoga','running','ski','surf'],
    'Fashion 👗':    ['fashion','style','outfit','couture','look','streetwear','moda','mode'],
    'Food 🍽️':      ['food','recipe','cuisine','chef','restaurant','coffee','café','baker'],
    'Travel ✈️':     ['travel','voyage','wanderlust','trip','explorer','adventure','viaje'],
    'Business 💼':   ['founder','ceo','entrepreneur','business','marketing','startup','consulting'],
    'Art 🎨':        ['art','artist','photo','photograph','design','creative','créat','arte'],
    'Music 🎵':      ['music','musician','dj','producer','song','album','musique'],
    'Tech 💻':       ['tech','developer','ai','crypto','web3','engineer','code','data'],
    'Media 📰':      ['news','media','journal','magazine','podcast','journalist'],
}

def detect_theme(bio):
    if not isinstance(bio, str) or not bio:
        return 'Other / Unclassified'
    bio_low = bio.lower()
    for theme, keywords in themes.items():
        if any(kw in bio_low for kw in keywords):
            return theme
    return 'Other / Unclassified'

df['theme'] = df['biography'].apply(detect_theme)
theme_counts = df['theme'].value_counts()
print('=== THEMATIC DISTRIBUTION ===')
print(theme_counts)

fig, ax = plt.subplots()
theme_counts.plot(kind='barh', ax=ax, color='#8e44ad')
ax.set_title('Dominant themes in follows', fontweight='bold')
ax.set_xlabel('Number of accounts')
plt.tight_layout()
plt.show()

## 6. 💎 Engagement analysis

In [ ]:
df_er = df_ok.dropna(subset=['last_post_engagement_rate_pct']).copy()
df_er['last_post_engagement_rate_pct'] = pd.to_numeric(df_er['last_post_engagement_rate_pct'], errors='coerce')
df_er['followers'] = pd.to_numeric(df_er['followers'], errors='coerce')
df_er = df_er.dropna(subset=['last_post_engagement_rate_pct','followers'])
df_er = df_er[df_er['followers'] > 100]

print('=== ENGAGEMENT RATE (%) ===')
print(df_er['last_post_engagement_rate_pct'].describe())

fig, ax = plt.subplots(figsize=(12,6))
ax.scatter(df_er['followers'], df_er['last_post_engagement_rate_pct'], alpha=0.4, s=15)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Followers (log)')
ax.set_ylabel('Engagement Rate % (log)')
ax.set_title('Followers vs Engagement Rate', fontweight='bold')
ax.axhline(1, color='green', linestyle='--', alpha=0.5, label='ER = 1% (normal)')
ax.axhline(3, color='blue', linestyle='--', alpha=0.5, label='ER = 3% (excellent)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
top_engagement = df_er[df_er['followers'] > 1000].sort_values('last_post_engagement_rate_pct', ascending=False).head(20)
print('=== 🌟 TOP 20 BEST ENGAGEMENT (>1k followers) ===')
top_engagement[['username','full_name','followers','last_post_engagement_rate_pct','days_since_last_post']]

In [ ]:
suspicious = df_er[(df_er['followers'] > 100000) & (df_er['last_post_engagement_rate_pct'] < 0.1)]
print(f'🚩 {len(suspicious)} accounts with >100k followers but ER < 0.1% (potentially low value) :')
suspicious[['username','full_name','followers','last_post_engagement_rate_pct','is_verified']].sort_values('followers', ascending=False).head(20)

## 7. 📤 Actionable exports

In [ ]:
from config import UNFOLLOW_CSV

to_unfollow = df[df['score_unfollow'] > 60].sort_values('score_unfollow', ascending=False)
to_unfollow[['username','full_name','days_since_last_post','followers','biography','last_post_engagement_rate_pct','score_unfollow']]\
    .to_csv(UNFOLLOW_CSV, index=False, encoding='utf-8-sig')
print(f'✅ {UNFOLLOW_CSV} exported : {len(to_unfollow)} accounts')

to_keep = df[(df['status']=='OK') & 
             (df['days_since_last_post'] < 30) & 
             (df['last_post_engagement_rate_pct'] > 2)]\
        .sort_values('last_post_engagement_rate_pct', ascending=False)
to_keep[['username','full_name','followers','last_post_engagement_rate_pct','theme']]\
    .to_csv('to_keep.csv', index=False, encoding='utf-8-sig')
print(f'✅ to_keep.csv exported : {len(to_keep)} accounts')

df.to_csv('follows_full_analysis.csv', index=False, encoding='utf-8-sig')
print(f'✅ follows_full_analysis.csv exported : {len(df)} rows')

## 8. 📋 Executive summary

In [ ]:
print('='*60)
print('📊 EXECUTIVE SUMMARY')
print('='*60)
print(f'\n📌 Total follows analyzed        : {len(df)}')
print(f'✅ Analyzable (OK)               : {(df["status"]=="OK").sum()}')
print(f'🔒 Private inaccessible          : {(df["status"]=="PRIVATE_NO_ACCESS").sum()}')
print(f'💀 Never posted                  : {(df["status"]=="NEVER_POSTED").sum()}')

df_ok_stats = df[df["status"]=="OK"]
days_series = pd.to_numeric(df_ok_stats["days_since_last_post"], errors='coerce').dropna()

print(f'\n📅 ACTIVITY')
print(f'  - Active (<7d)         : {(days_series<=7).sum()}')
print(f'  - Semi-active (7-90d)  : {((days_series>7)&(days_series<=90)).sum()}')
print(f'  - Dormant (90-365d)    : {((days_series>90)&(days_series<=365)).sum()}')
print(f'  - Dead (>1 year)       : {(days_series>365).sum()}')

print(f'\n🎯 RECOMMENDATIONS')
print(f'  - To unfollow (score>60)      : {(df["score_unfollow"]>60).sum()}')
print(f'  - Never posted (obvious)      : {(df["status"]=="NEVER_POSTED").sum()}')

print(f'\n🏷️ TOP 3 THEMES')
for theme, count in df["theme"].value_counts().head(3).items():
    print(f'  - {theme:25s} : {count}')

print('\n' + '='*60)
print('📁 Exported files :')
print(f'  - {UNFOLLOW_CSV}           (unfollow candidates)')
print('  - to_keep.csv                (excellent accounts)')
print('  - follows_full_analysis.csv  (enriched dataset)')
print('='*60)